# Clase 128 — Capas convolucionales, filtros, feature maps

La **convolución 2D** desliza un filtro `K×K` de pesos aprendibles sobre la
imagen y produce un **feature map**. Frente a un MLP, una CNN es
*parameter-efficient* gracias a tres ideas: **parameter sharing** (el mismo
filtro en toda la imagen), **localidad** (cada neurona mira una vecindad) e
**invariancia a translación**. Apilando capas, la red aprende jerarquías
visuales: bordes → texturas → partes → objetos.

Requiere: `numpy`, `scipy`, `tensorflow` / `keras` (≥ 3.0). TF/Keras no se
ejecuta aquí; el código es idiomático y correcto por API.

### 🧠 Intuición previa

**Una convolución es un filtro que se desliza por la imagen buscando el mismo patrón en todas partes.**

Imaginá una pequeña plantilla de `3×3` — por ejemplo, un detector de 'borde vertical'. La convolución la desliza (*slide*) por toda la imagen y, en cada posición, mide cuánto se parece el trozo de imagen a la plantilla. Donde hay un borde vertical, responde fuerte; donde no, responde débil. El resultado de barrer toda la imagen con un filtro es un **feature map**: un mapa de '¿dónde aparece este patrón?'.

Dos consecuencias clave:

- **Parameter sharing**: es el *mismo* filtro (los mismos ~9 pesos) en toda la imagen. Por eso una CNN tiene muchísimos menos parámetros que un MLP y sus parámetros **no dependen del tamaño de la imagen**.
- **Invariancia a translación**: si el patrón aparece en otra posición, el mismo filtro lo detecta igual.

La red **aprende** los filtros: las primeras capas descubren bordes y colores, y al apilar capas se combinan en texturas, partes y objetos.

## 1. Convolución 2D a mano con `scipy.signal.convolve2d`

In [ ]:
import numpy as np
from scipy.signal import convolve2d

# Imagen sintética 6x6 con un borde vertical (mitad derecha encendida)
img = np.zeros((6, 6), dtype=np.float32)
img[:, 3:] = 1.0

# Kernel Sobel-x: detecta bordes verticales
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

# 'valid' (sin padding) reduce el tamaño; 'same' lo preserva con zeros
fmap_valid = convolve2d(img, sobel_x, mode="valid")
fmap_same = convolve2d(img, sobel_x, mode="same")
print("entrada:", img.shape, "-> valid:", fmap_valid.shape,
      "| same:", fmap_same.shape)
print("feature map (same): la respuesta se concentra en el borde")
print(np.round(fmap_same, 1))

## 2. `Conv2D` en Keras: `padding='same'` vs `'valid'`

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

entrada = keras.Input(shape=(28, 28, 1))
same = layers.Conv2D(8, kernel_size=3, strides=1,
                     padding="same", activation="relu")(entrada)
valid = layers.Conv2D(8, kernel_size=3, strides=1,
                      padding="valid", activation="relu")(entrada)
print("entrada         : (28, 28, 1)")
print("padding='same'  :", same.shape)    # (None, 28, 28, 8) -> preserva HxW
print("padding='valid' :", valid.shape)   # (None, 26, 26, 8) -> reduce

## 3. Número de parámetros de una `Conv2D`

Una conv con kernel `K×K`, `C_in` canales de entrada y `F` filtros tiene
`K*K*C_in*F + F` parámetros (los bias son `F`).

In [ ]:
def conv_params(kernel, c_in, filters):
    return kernel * kernel * c_in * filters + filters

# Conv2D(32, kernel_size=5) sobre entrada (28, 28, 1)
print("5*5*1*32 + 32 =", conv_params(5, 1, 32))   # 832

modelo = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 5, padding="same", activation="relu"),
])
print("Keras count_params:", modelo.count_params())   # coincide: 832

## 4. Stride y shape de salida

La fórmula general es `H' = floor((H - K + 2P) / S) + 1`. Con `strides=2` el
filtro salta de a 2 píxeles y hace *subsampling* (halve H y W).

In [ ]:
entrada = keras.Input(shape=(28, 28, 1))
s1 = layers.Conv2D(32, 3, strides=1, padding="same")(entrada)
s2 = layers.Conv2D(32, 3, strides=2, padding="same")(entrada)
print("strides=1 ->", s1.shape)   # (None, 28, 28, 32)
print("strides=2 ->", s2.shape)   # (None, 14, 14, 32) subsampling

def out_dim(H, K, P, S):
    return (H - K + 2 * P) // S + 1

print("valid K=3 S=1:", out_dim(28, 3, 0, 1))   # 26
print("valid K=3 S=2:", out_dim(28, 3, 0, 2))   # 13

## 5. Feature maps: activaciones intermedias

Un feature map es la salida de un filtro sobre toda la imagen. Con un modelo
que expone la salida de una capa `Conv` intermedia extraemos sus activaciones.

In [ ]:
mini = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu", name="conv1"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation="softmax"),
])
extractor = keras.Model(inputs=mini.input,
                        outputs=mini.get_layer("conv1").output)
img = np.random.rand(1, 28, 28, 1).astype("float32")
fmaps = extractor.predict(img, verbose=0)
print("feature maps de conv1:", fmaps.shape)   # (1, 28, 28, 16): 16 mapas

## 6. Filtros aprendidos y eficiencia paramétrica

El kernel de la primera capa es el *banco de filtros* aprendido. Además, una
`Conv2D` con parameter sharing usa muchísimos menos pesos que un `Dense`
equivalente que mirara todos los píxeles.

In [ ]:
kernels = mini.get_layer("conv1").kernel   # shape (3, 3, 1, 16)
print("banco de filtros:", kernels.shape)
print("primer filtro 3x3:")
print(np.round(kernels.numpy()[:, :, 0, 0], 2))

conv = keras.Sequential([keras.Input(shape=(28, 28, 1)),
                         layers.Conv2D(32, 3, padding="same")])
dense = keras.Sequential([keras.Input(shape=(28 * 28,)),
                          layers.Dense(32 * 28 * 28)])
print("Conv2D(32, 3) params :", conv.count_params())    # 320 (comparte pesos)
print("Dense equivalente    :", dense.count_params())   # ~19.6M

## Ejercicios

1. **Conv básica**: aplicá `Conv2D(8, 3, padding='same', activation='relu')` a
   una entrada `(28, 28, 1)` y verificá que la salida es `(batch, 28, 28, 8)`.
2. **Conteo de parámetros**: comprobá que `Conv2D(32, kernel_size=5)` sobre
   `(28, 28, 1)` tiene `5*5*1*32 + 32 = 832` parámetros.
3. **Stride 2**: con `Conv2D(32, 3, strides=2, padding='same')` verificá que la
   salida reduce H y W a la mitad.
4. **Feature maps**: extraé las activaciones de la primera Conv de la mini-CNN
   para una imagen y contá cuántos mapas produce.

## Conclusiones

- La convolución 2D desliza un filtro `K×K` y produce un feature map por filtro.
- El shape de salida sigue `H' = (H - K + 2P)/S + 1`; `'same'` preserva, `'valid'` reduce.
- Los parámetros de una conv son `K*K*C_in*F + F`, independientes del tamaño de la imagen.
- Parameter sharing + localidad hacen a las CNN mucho más eficientes que un MLP.
- Apilando convs la red aprende jerarquías: bordes → texturas → partes → objetos.

## ✅ Soluciones de los ejercicios

Capas convolucionales, filtros y feature maps en Keras 3 (cap. 14). API real (se valida por AST sin TF); el **Ej. 2** (conteo de parámetros) es NumPy puro con `assert`. Cubren shape de salida, parámetros independientes del tamaño de imagen, stride, extracción de feature maps y visualización de filtros aprendidos.

**Ej. 1 — Conv básica.** `Conv2D(8, 3, padding='same')` sobre `(28,28,1)` -> `(batch,28,28,8)`.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D

x = tf.random.normal((4, 28, 28, 1))
y = Conv2D(8, 3, padding="same", activation="relu")(x)
print(y.shape)     # (4, 28, 28, 8): 'same' conserva H,W; salen 8 feature maps

**Ej. 2 — Conteo de parámetros.** `k·k·in·out + out`; NumPy puro con `assert` (no depende del tamaño de la imagen).

In [ ]:
def conv_params(out_ch, k, in_ch):
    return k * k * in_ch * out_ch + out_ch      # +out_ch por los bias

p = conv_params(out_ch=32, k=5, in_ch=1)
print("Conv2D(32, kernel_size=5) sobre (28,28,1):", p, "params")
assert p == 832                                  # 5*5*1*32 + 32 = 832
# clave: los params NO dependen de H,W de la imagen (parameter sharing)

**Ej. 3 — Stride 2.** `Conv2D(32, 3, strides=2, padding='same')` -> mitad de resolución.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D

x = tf.random.normal((4, 28, 28, 1))
y = Conv2D(32, 3, strides=2, padding="same")(x)
print(y.shape)     # (4, 14, 14, 32): stride 2 con 'same' -> ceil(H/2)

**Ej. 4 — Visualizar feature maps.** Modelo auxiliar que expone las activaciones de la 1a conv.

In [ ]:
import tensorflow as tf
from tensorflow import keras

cnn = keras.Sequential([keras.Input((28, 28, 1)),
                        keras.layers.Conv2D(16, 3, activation="relu"),
                        keras.layers.Conv2D(32, 3, activation="relu"),
                        keras.layers.GlobalAveragePooling2D(),
                        keras.layers.Dense(10, activation="softmax")])
# extractor de activaciones intermedias:
feat_model = keras.Model(cnn.input, cnn.layers[0].output)
maps = feat_model(tf.random.normal((1, 28, 28, 1)))
print("feature maps de la 1a conv:", maps.shape)   # (1, 26, 26, 16)
# plt.imshow(maps[0, :, :, k], cmap="viridis")  para el canal k

**Ej. 5 — Filtros aprendidos.** Leer `layer.kernel.numpy()`; tras entrenar, los primeros filtros ven bordes/colores.

In [ ]:
import tensorflow as tf
from tensorflow import keras

cnn = keras.Sequential([keras.Input((32, 32, 3)),
                        keras.layers.Conv2D(16, 3, activation="relu")])
kernels = cnn.layers[0].kernel.numpy()   # shape (3, 3, 3, 16): (kh, kw, in_ch, out_ch)
print("kernels shape:", kernels.shape)
# tras entrenar en CIFAR, los primeros filtros aprenden bordes y manchas de color:
# f0 = kernels[:, :, :, 0]; plt.imshow((f0 - f0.min()) / (f0.ptp() + 1e-9))